In [ ]:
import xarray as xr
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import cmocean as cmo
import cmcrameri as cmc
from scipy import stats
from scipy.optimize import curve_fit
from scipy.stats import binned_statistic_2d

import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
ds = xr.open_dataset('../input/Laskar_etal_2004_insolation_moncent.nc')
Q_TOA = ds.Q_TOA.isel(time=-1).drop_vars('time')
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/mltgl_*.nc')
melt = ds.mltgl.isel(height=0)
ds.close()

# Interpolate Q to data array
Q_interp = Q_TOA.interp(lat=melt['lat'])
month_indices = melt.coords['time'].dt.month.values - 1
Q_ins = Q_interp.isel(month=('time', month_indices)).transpose('time', 'rlat', 'rlon')

In [ ]:
ds = xr.open_dataset('/home/erwin/data/dalum_racmo241/ANT11_masks.nc')
mask = ds.IceMask
topo = ds.Topography.values.squeeze()
lat = ds.lat.values.squeeze()
lon = ds.lon.values.squeeze()
ds.close()

In [ ]:
lon_rad = np.radians(ds.lon.values)
lat_rad = np.radians(ds.lat.values)
R = 6371000

# --- Physical spacing ---
# y-spacing: change in physical distance along the rlat dimension (axis=0)
dlat_rad = np.abs(np.gradient(np.deg2rad(lat), axis=0))  # (591, 726)
dy = R * dlat_rad                                       # (591, 726) — meters per grid step

# x-spacing: change in physical distance along the rlon dimension (axis=1)
dlon_rad = np.abs(np.gradient(np.deg2rad(lon), axis=1))   # (591, 726)
cos_lat = np.cos(np.deg2rad(lat))                        # (591, 726)
dx = R * cos_lat * dlon_rad                              # (591, 726) — meters per grid step

# --- Gradient in grid-index space ---
dz_dy, dz_dx = np.gradient(topo, axis=(0, 1))            # (591, 726) each

# --- Absolute slope (nondimensional, m/m) ---
slope = np.sqrt(dz_dy**2 + dz_dx**2)  # (591, 726)

ref = melt
Hs_slope = xr.DataArray(
    data=np.broadcast_to(slope[None, :, :], ref.shape),  # Add time axis
    dims=['time', 'rlat', 'rlon'],
    coords={
        'time': ref.time.values,
        'rlat': ref.rlat.values,
        'rlon': ref.rlon.values,
    },
    attrs={
        'long_name': 'Topography slope',
        'units': 'dimensionless',
        'standard_name': 'surface_slope'
    }
)

Hs_slope = Hs_slope.chunk({'time':1})

## Melt

In [ ]:
ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/mltgl_*.nc')
melt = ds.mltgl.isel(height=0)
lat = ds.lat
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/tas_*.nc')
tas = ds.tas.isel(height=0)
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/rsds_*.nc')
Qswd = ds.rsds.isel(height=0)
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/rsusgl_*.nc')
Qswu = ds.rsusgl.isel(height=0)
ds.close()

melt = melt.where(mask,np.nan).values / 1000 # kg/m2/month to mwe/month
tas = tas.where(mask,np.nan).values
albedo = (Qswu/Qswd).where(mask,np.nan).values
albedo = np.maximum(0,np.minimum(1,albedo))
Qnet = ((1-albedo) * Q_ins).values # W m^-2

valid = ~np.isnan(melt) & ~np.isnan(tas) & ~np.isnan(Qnet) & ~np.isinf(Qnet) & (melt>0.01)
melt_valid = melt[valid]
tas_valid = tas[valid]
Qnet_valid = Qnet[valid]


In [ ]:
xbins = np.linspace(260, tas_valid.max(), 50)
ybins = np.linspace(Qnet_valid.min(), Qnet_valid.max(), 50)

statistic, x_edges, y_edges, binnumber = binned_statistic_2d(
    tas_valid, Qnet_valid, melt_valid,
    statistic='mean',  # or 'mean', 'sum', 'std'
    bins=[xbins, ybins],
    expand_binnumbers=False
)

fig, ax = plt.subplots(1,3,figsize=(15,5))
mesh = ax[0].pcolormesh(
    x_edges, y_edges,
    statistic.T,  # Transpose to match axis orientation
    cmap='Reds',
)
ax[0].contour(x_edges[:-1],y_edges[:-1],statistic.T,20,linewidths=.2,colors='k')
plt.colorbar(mesh)

cmap = plt.get_cmap('cmo.thermal')
for y,yy in enumerate(ybins[:-1]):
    ax[1].plot(xbins[:-1],statistic[:,y],c=cmap((yy-ybins[0])/(ybins[-1]-ybins[0])))

for x,xx in enumerate(xbins[:-1]):
    ax[2].plot(ybins[:-1],statistic[x,:],c=cmap((xx-xbins[0])/(xbins[-1]-xbins[0])))

ax[0].set_xlabel('T2m')
ax[0].set_ylabel('(1-a) Q')
ax[1].set_xlabel('T2m')
ax[2].set_xlabel('(1-a) Q')
ax[1].set_ylabel('Melt')
ax[2].set_ylabel('Melt')

In [ ]:
def func(data, a, b, c):
    T,Q = data
    return a*np.maximum(0,(T-c))**2 + b*Q*np.maximum(0,T-c)

popt, pcov = curve_fit(func, 
    (tas_valid, Qnet_valid), 
    melt_valid, 
    p0=[100, 0.2, 260])
a, b, c = popt

print(f"C_melt_temp_quad = {a:.6f} mwe K^-2")
print(f"C_melt_insol = {b:.9f} mwe (Wm^-2)^-1")
print(f"C_trans_temp = {c:.2f} K")

melt_pred = func((tas_valid,Qnet_valid),*popt)

In [ ]:
fig,ax = plt.subplots(1,3, figsize=(10,3),sharey=True)

h, xedges, yedges, im = ax[0].hist2d(
    tas_valid, melt_valid,
    bins=(100, 100),          # Resolution
    cmap='viridis',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

h, xedges, yedges, im = ax[1].hist2d(
    Qnet_valid, melt_valid,
    bins=(100, 100),          # Resolution
    cmap='viridis',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

h, xedges, yedges, im = ax[2].hist2d(
    melt_pred, melt_valid,
    bins=(100, 100),          # Resolution
    cmap='viridis',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

ax[2].plot([0,2000],[0,2000],c='r',lw=1)

ax[0].set_ylabel('Melt RACMO')
ax[0].set_xlabel('Tas')
ax[1].set_xlabel('(1-a)t Q')
ax[2].set_xlabel('Melt pred F(Tas, (1-a)t Q)')


In [ ]:
ds = xr.open_dataset('/home/erwin/data/dalum_racmo241/mltgl_monthlyS_ANT11_RACMO2.4p1_ERA5_197901_202312.nc')
melt = ds.mltgl.isel(height=0).mean(dim='time')*12. # mm w.e. /y
ds.close()

ds = xr.open_dataset('../hpcoutput/t2p5/main_output_ANT_grid.nc')
M = ds.SurfaceMelt.sum(dim='month').isel(time=-1) * 1000 # Integrate to mm w.e./y
umask = ds.Hi.isel(time=-1).values>0
ux = ds.x.values
uy = ds.y.values
ds.close()


data_crs = ccrs.PlateCarree()
proj = ccrs.Stereographic(central_latitude=-90, central_longitude=0)

fig,ax = plt.subplots(1,3,figsize=(11,3))

h, xedges, yedges, im = ax[0].hist2d(
    melt_pred*1000., melt_valid*1000.,
    bins=(100, 100),
    cmap='afmhot_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6)
)
ax[0].plot([0,2500],[0,2500],c='r',lw=1)
ax[0].set_xlabel('F(Tas, Q_net) [mmwe/month]')
ax[0].set_ylabel('Melt RACMO [mmwe/month]')
ax[0].set_title('Melt')

plt.colorbar(im,ax=ax[0],label='Counts')

bounds = [0,10,30,50,100,200,500,1000,2000]
cmap = plt.get_cmap('Spectral_r',len(bounds))
norm = mpl.colors.BoundaryNorm(bounds,ncolors=len(bounds)-1)

ax1 = fig.add_subplot(1,3,2,projection=proj)
ax1.set_extent([-180,180,-90,-45], crs=data_crs)
im = ax1.pcolormesh(lon,lat,np.where(mask,melt,np.nan),cmap=cmap,norm=norm, transform=data_crs)
im = ax[2].pcolormesh(ux,uy,np.where(umask,M,np.nan),cmap=cmap,norm=norm)
plt.colorbar(im,ax=ax[2],label='Melt [mm w.e. / yr]',extend='max')
ax1.set_title('RACMO: 123.7 Gt/y')
Mtot = np.nansum(M)*16*16*1e-6
ax[2].set_title(f'ITM: {Mtot:.1f} Gt/y')

for Ax in ax:
    Ax.set_aspect(1)

for Ax in [ax1,ax[2]]:
    Ax.set_xlim([-3e6,3e6])
    Ax.set_ylim([-3e6,3e6])

for Ax in ax[1:]:
    Ax.set_xticks([])
    Ax.set_yticks([])

plt.tight_layout()
plt.savefig('../figures/draftplot_ITM_melt.png',dpi=600)

## Insolation

In [ ]:
fig,ax = plt.subplots(1,1)
ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/rsds_*.nc')
Qswd = ds.rsds.isel(height=0) / (3600*24*30)
ds.close()

ax.plot(Q_ins.isel(rlat=300,rlon=350).isel(time=slice(0,36)))
ax.plot(Qswd.isel(rlat=300,rlon=350).isel(time=slice(0,36)))

In [ ]:
ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/rsds_*.nc')
Qswd = ds.rsds.isel(height=0) / (3600*24*30)
ds.close()

Hs = xr.DataArray(
    data=np.broadcast_to(topo[None, :, :], Qswd.shape),  # Add time axis
    dims=['time', 'rlat', 'rlon'],
    coords={
        'time': Qswd.time.values,
        'rlat': Qswd.rlat.values,
        'rlon': Qswd.rlon.values,
    }
)
Hs = Hs.chunk({'time':1}).values

Qswd = Qswd.where(mask,np.nan).values

valid = ~np.isnan(Qswd) & ~np.isnan(Hs)

Hs_valid = Hs[valid]
Qswd_valid = Qswd[valid]
Qins_valid = Q_ins.values[valid]

def func(data,a,b):
    H,Qtoa = data
    transm = a + b*H
    return transm * Qtoa

popt, pcov = curve_fit(func, (Hs_valid,Qins_valid),Qswd_valid, p0=[.8,.001])
print(popt)
Qswd_pred = func((Hs_valid, Qins_valid),*popt)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,5))

h, xedges, yedges, im = ax.hist2d(
    Qswd_pred, Qswd_valid,
    bins=(300, 300),          # Resolution
    cmap='cmc.batlow_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

ax.plot([0,500],[0,500],c='r')

## Albedo

In [ ]:
ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/mltgl_*.nc')
melt = ds.mltgl.isel(height=0)
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/rsds_*.nc')
Qswd = ds.rsds.isel(height=0)
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/rsusgl_*.nc')
Qswu = ds.rsusgl.isel(height=0)
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/sf_*.nc')
sf = ds.sf.isel(height=0)
ds.close()

melt = melt.where(mask,np.nan).values[:-1,:,:] / 1000 # kg/m2/month to mwe/month
sf = sf.where(mask,np.nan).values[1:,:,:]
albedo = (Qswu/Qswd).where(mask,np.nan).values[1:,:,:]
albedo = np.maximum(0,np.minimum(1,albedo))

valid = ~np.isnan(melt) & ~np.isnan(albedo) & ~np.isinf(albedo) & ~np.isnan(sf)
melt_valid = melt[valid]
sf_valid = sf[valid]
albedo_valid = albedo[valid]


In [ ]:
xbins = np.linspace(sf_valid.min(), sf_valid.max(), 50)
ybins = np.linspace(0, melt_valid.max(), 50)

statistic, x_edges, y_edges, binnumber = binned_statistic_2d(
    sf_valid, melt_valid, albedo_valid,
    statistic='mean',  # or 'mean', 'sum', 'std'
    bins=[xbins, ybins],
    expand_binnumbers=False
)

fig, ax = plt.subplots(1,3,figsize=(15,5))
mesh = ax[0].pcolormesh(
    x_edges, y_edges,
    statistic.T,  # Transpose to match axis orientation
    cmap='Reds',
)
ax[0].contour(x_edges[:-1],y_edges[:-1],statistic.T,20,linewidths=.2,colors='k')
plt.colorbar(mesh)

cmap = plt.get_cmap('cmo.thermal')
for y,yy in enumerate(ybins[:-1]):
    ax[1].plot(xbins[:-1],statistic[:,y],c=cmap((yy-ybins[0])/(ybins[-1]-ybins[0])))

for x,xx in enumerate(xbins[:-1]):
    ax[2].plot(ybins[:-1],statistic[x,:],c=cmap((xx-xbins[0])/(xbins[-1]-xbins[0])))

ax[0].set_xlabel('sf')
ax[0].set_ylabel('melt')
ax[1].set_xlabel('sf')
ax[2].set_xlabel('melt')
ax[1].set_ylabel('albedo')
ax[2].set_ylabel('albedo')

In [ ]:
def func(data,a,b,c):
    M,S = data
    a0 = np.maximum(.45,.85+a*M)
    return b-(b-a0)*np.exp(-S/c)

popt, pcov = curve_fit(func, (melt_valid,sf_valid),albedo_valid, p0=[-.01,.95,100])
print(popt)
albedo_pred = func((melt_valid, sf_valid),*popt)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,4))

h, xedges, yedges, im = ax[0].hist2d(
    Qswd_pred, Qswd_valid,
    bins=(300, 300),          # Resolution
    cmap='afmhot_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

ax[0].plot([0,500],[0,500],c='r')

h, xedges, yedges, im = ax[1].hist2d(
    albedo_pred, albedo_valid,
    bins=(300, 300),          # Resolution
    cmap='afmhot_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6),
    vmin=None               # Auto-scaling
)

ax[1].plot([0,1],[0,1],c='r')

ax[1].set_xlim([.4,1])
ax[1].set_ylim([.4,1])
for Ax in ax:
    Ax.set_aspect(1)


ax[0].set_xlabel('F(Q_TOA,a) [W/m2]')
ax[0].set_ylabel('Q_net [W/m2]')

ax[1].set_xlabel('F(M,S) []')
ax[1].set_ylabel('Albedo []')

plt.colorbar(im,ax=ax[1],label='Count')

plt.tight_layout()
plt.savefig('../figures/draftplot_Q_alpha.png',dpi=600)

## Snow fraction

In [ ]:
ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/tas_*.nc')
tas = ds.tas.isel(height=0).values
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/pr_*.nc')
pr = ds.pr.isel(height=0)
ds.close()

ds = xr.open_mfdataset('/home/erwin/data/hofsteenge/sf_*.nc')
sf = ds.sf.isel(height=0)
ds.close()

snowfrac = xr.where(pr>0,sf/pr,np.nan)
snowfrac = snowfrac.where(mask>0, np.nan)
snowfrac = snowfrac.where(snowfrac<1,np.nan).values

valid = ~np.isnan(snowfrac) & ~np.isnan(tas)

snowfrac_valid = snowfrac[valid]
tas_valid = tas[valid]


In [ ]:
def func_atan(x, a, b, c):
    return np.maximum(0,np.minimum(1,a*(1- np.arctan((x-273.16) / b) / c)))

def func_tanh(x, a, b):
    return 0.5 * (1 - np.tanh((x-a)/b))

popt1, pcov1 = curve_fit(func_atan, tas_valid, snowfrac_valid, p0=[.5, 3.5, 1.2])
print(popt1)
popt2, pcov2 = curve_fit(func_tanh, tas_valid, snowfrac_valid, p0=[273.16,2])
print(popt2)

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(10,3),sharey=True)

# 2D histogram
h, xedges, yedges, im = ax[0].hist2d(
    tas_valid, snowfrac_valid,
    bins=(100, 100),          # Resolution
    cmap='afmhot_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6)
)

xarr = np.linspace(250,290,100)
yarr1 = func_atan(xarr,*popt1)
yarr2 = func_tanh(xarr,*popt2)
ax[0].plot(xarr,func_atan(xarr,*popt1),c='r',lw=3,label='Fit atan')
ax[0].plot(xarr,func_tanh(xarr,*popt2),c='k',lw=3,label='Fit tanh')
ax[0].plot(xarr,func_atan(xarr,.5,3.5,1.25664),c='y',label='IMAU-ice?')
ax[0].plot(xarr,func_atan(xarr,.725,5.95,1.8566),c='m',label='ANICE?')

h, xedges, yedges, im = ax[1].hist2d(
    func_atan(tas_valid,*popt1), snowfrac_valid,
    bins=(100, 100),          # Resolution
    cmap='afmhot_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6)
)

h, xedges, yedges, im = ax[2].hist2d(
    func_tanh(tas_valid,*popt2), snowfrac_valid,
    bins=(100, 100),          # Resolution
    cmap='afmhot_r',
    norm = mpl.colors.LogNorm(vmin=1,vmax=1e6)
)

ax[0].legend()
ax[0].set_xlim([250,290])
ax[0].set_xlabel('Tas')
ax[0].set_ylabel('Snow fraction')

for Ax in ax:
    Ax.set_ylim([0,1])

ax[1].plot([0,1],[0,1],c='r')
ax[2].plot([0,1],[0,1],c='k')

ax[1].set_xlabel('Fit atan')
ax[2].set_xlabel('Fit tanh')

plt.tight_layout()
plt.savefig('../figures/draftplot_ITM_snowfrac.png',dpi=600)

## Firn Air Content diagnosis

In [ ]:
ds = xr.open_dataset('../hpcoutput/t2p5/main_output_ANT_grid.nc')
FAC = ds.FirnAirContent.isel(month=-1)
ds.close()
FAC = FAC.where(FAC > 0, np.nan)
FACmean = np.nanmean(FAC,axis=(1,2))

fig,ax = plt.subplots(1,2,figsize=(10,4))
img = mpl.image.imread('../figures/FAC_Veldhuisen2.png')
ax[0].imshow(img)

im = ax[1].pcolormesh(FAC.isel(time=-1),cmap='cmc.batlow',vmin=0,vmax=40)
plt.colorbar(im,ax=ax[0],label='Firn air content [m]')

ax[0].set_title('IMAU-FDM: 22.8 m')
ax[1].set_title(f'ITM: {FACmean[-1]:.1f} m')

for Ax in ax:
    Ax.axis('off')
    Ax.set_aspect(1)

plt.tight_layout()
plt.savefig('../figures/draftplot_ITM_FAC.png',dpi=600)

## SMB

In [ ]:
ds = xr.open_dataset('/home/erwin/data/dalum_racmo241/smbgl_monthlyS_ANT11_RACMO2.4p1_ERA5_197901_202312.nc')
smb = ds.smbgl.isel(height=0).mean(dim='time')*12. # mm w.e. /y
ds.close()

ds = xr.open_dataset('../hpcoutput/t2p5/main_output_ANT_grid.nc')
M = ds.SMB.isel(time=-1) * 918 # Integrate to mm w.e./y

ds.close()

fig,ax = plt.subplots(1,2,figsize=(12,4))

bounds = [-50,0,10,30,50,100,200,500,1000,2000]
cmap = plt.get_cmap('RdYlBu_r',len(bounds))
norm = mpl.colors.BoundaryNorm(bounds,ncolors=len(bounds)-1)

ax1 = fig.add_subplot(1,2,1,projection=proj)
ax1.set_extent([-180,180,-90,-45], crs=data_crs)
im = ax1.pcolormesh(lon,lat,np.where(mask,smb,np.nan),cmap=cmap,norm=norm, transform=data_crs)

im = ax[1].pcolormesh(ux,uy,np.where(umask,M,np.nan),cmap=cmap,norm=norm)
plt.colorbar(im,ax=ax[1],label='SMB [mm w.e. / yr]',extend='max')
ax[0].set_title('RACMO: 2546 Gt/y')
Mtot = np.nansum(M)*16*16*1e-6
ax[1].set_title(f'ITM: {Mtot:.0f} Gt/y')

for Ax in ax:
    Ax.set_aspect(1)

for Ax in [ax1,ax[1]]:
    Ax.set_xlim([-3e6,3e6])
    Ax.set_ylim([-3e6,3e6])

for Ax in ax:
    Ax.set_xticks([])
    Ax.set_yticks([])

plt.tight_layout()
plt.savefig('../figures/draftplot_ITM_smb.png',dpi=600)